# Credit Card Customer Segmentation
>  **Unsupervised Machine Learning | PCA + Gaussian Mixture Model**

This notebook walks through the complete analysis pipeline:
1. Data exploration & preprocessing
2. Dimensionality reduction (PCA)
3. Cluster count selection via Silhouette analysis
4. Final GMM clustering
5. Cluster profiling & business insights

---
**Dataset**: 8,950 credit card customers · 17 behavioural features  
**Best model**: GMM with **7 clusters** · Silhouette score **0.4671**

## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.utils import set_plot_style
set_plot_style()

DATA_PATH = "../data/CC_GENERAL.csv"  # update if needed
print("Setup complete.")

---
## 1 · Data Exploration

In [ ]:
from src.preprocessing import load_data

df_raw = load_data(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

In [ ]:
missing = df_raw.isnull().sum()
print("Missing values:")
print(missing[missing > 0])

In [ ]:
key_features = ["BALANCE","PURCHASES","CASH_ADVANCE","INSTALLMENTS_PURCHASES","CREDIT_LIMIT","PAYMENTS"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, key_features):
    df_raw[feat].dropna().hist(ax=ax, bins=50, color="#4C72B0", edgecolor="white")
    ax.set_title(feat, fontsize=10)
fig.suptitle("Feature Distributions Before Log Transformation", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2 · Preprocessing

In [ ]:
from src.preprocessing import handle_missing_values, apply_log_transform, scale_features

df_imputed = handle_missing_values(df_raw)
df_log     = apply_log_transform(df_imputed)
X_scaled, scaler = scale_features(df_log)
print(f"Scaled matrix shape: {X_scaled.shape}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, key_features):
    df_log[feat].dropna().hist(ax=ax, bins=50, color="#55A868", edgecolor="white")
    ax.set_title(feat, fontsize=10)
fig.suptitle("Feature Distributions After Log Transformation", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df_log.corr(), annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 7})
ax.set_title("Feature Correlation Matrix (Log-Transformed)")
plt.tight_layout()
plt.show()

---
## 3 · Dimensionality Reduction — PCA

In [ ]:
from src.dimensionality_reduction import fit_pca, select_n_components_by_variance, plot_explained_variance

n_opt = select_n_components_by_variance(X_scaled, threshold=0.995)
print(f"Optimal components for 99.5% variance: {n_opt}")

In [ ]:
X_pca, pca = fit_pca(X_scaled, n_components=12)
plot_explained_variance(pca)

> **Result**: 12 principal components retain **99.52%** of total variance.

---
## 4 · Determining Optimal Number of Clusters

In [ ]:
from src.clustering import find_optimal_clusters, plot_silhouette_scores

print("Evaluating GMM for n = 2 to 10 clusters ...")
scores = find_optimal_clusters(X_pca)
plot_silhouette_scores(scores)

> **Decision**: **7 clusters** achieved the highest Silhouette Score of **0.46710**.

---
## 5 · Final GMM Clustering

In [ ]:
from src.clustering import fit_gmm, build_cluster_summary, print_cluster_profiles

labels, gmm = fit_gmm(X_pca, n_components=7)
df_clean = df_imputed.copy()
df_clean["CLUSTER"] = labels

---
## 6 · Cluster Visualisation

In [ ]:
from src.visualization import plot_tsne, plot_cluster_heatmap, plot_cluster_distribution
from src.clustering import CLUSTER_PROFILES

cluster_names = {k: v["name"] for k, v in CLUSTER_PROFILES.items()}
plot_cluster_distribution(labels, cluster_names=cluster_names)

In [ ]:
plot_tsne(X_pca, labels, cluster_names=cluster_names)

In [ ]:
summary = build_cluster_summary(df_imputed, labels)
plot_cluster_heatmap(summary)

---
## 7 · Cluster Profiles & Business Insights

In [ ]:
print_cluster_profiles(labels)

In [ ]:
summary

### Business Insights Summary

| Cluster | Name | Strategy |
|---------|------|----------|
| 0 | Heavy Cash Advance Users | Risk monitoring; installment conversion offers |
| 1 | Installment-Oriented Customers | Zero-interest campaigns; loyalty rewards |
| 2 | Cash + Installment Mixed Users | Debt consolidation; enhanced risk tracking |
| 3 | Cash Advance Only Customers | Limit cash advance exposure; financial advisory |
| 4 | High Spenders (No Cash Advance) | Premium cards; personalised cashback rewards |
| 5 | Super Active All-Channel Users | VIP programmes; cross-sell investment products |
| 6 | Purchase-Only Customers | Encourage installments; digital loyalty enrolment |

---
## 8 · Save Results

In [ ]:
from src.utils import save_results, save_model

save_results(df_imputed, labels, path="../outputs/segmented_customers.csv")
save_model(gmm,    "../outputs/gmm_model.pkl")
save_model(pca,    "../outputs/pca_model.pkl")
save_model(scaler, "../outputs/scaler.pkl")
print("All artefacts saved.")

---
## Conclusion

The GMM segmentation successfully grouped **8,950 credit card customers** into **7 distinct behavioural segments**:

- **Risk management**: Clusters 0, 2, and 3 require closer credit monitoring.
- **Revenue growth**: Clusters 4 and 5 are prime candidates for premium product cross-sells.
- **Engagement**: Cluster 6 customers can be nudged toward higher-value behaviours through targeted campaigns.

Customer segmentation transforms raw transactional data into actionable strategic intelligence.